# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/0mneeha93/ML-Track/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method: K-Means clustering**, per the training-honest-models skill's guidance for "grouping items" tasks: pick k using silhouette score, then name clusters only after inspecting what's actually inside them never before.

This fits Lane 3 (Structured Content Archetype Clustering) directly there's no observed label to predict, so classification methods (Logistic Regression, Random Forest) don't apply here. K-Means is the standard, explainable starting point for unsupervised grouping: cluster centers are simple numbers to interpret, unlike more complex clustering methods.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

K-Means doesn't need a train/test split in the supervised sense there's no label to overfit to. Instead, I'll check cluster **stability**: fit K-Means on a random 70% subset of March 2026 pages, then refit on the full set, and confirm the cluster profiles look similar. If clusters shift wildly between the subset and full data, that's a sign the clustering isn't finding a real, stable pattern.

In [17]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

!pip install -q duckdb huggingface_hub
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("DuckDB ready.")

DuckDB ready.


In [18]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features_df = con.execute("""
    SELECT
        f.content_hash_id,
        AVG(f.gsc_avg_position) as avg_position,
        SUM(f.gsc_impressions) as impressions_total,
        CASE WHEN SUM(f.gsc_impressions) > 0 THEN SUM(f.gsc_clicks)*1.0/SUM(f.gsc_impressions) ELSE 0 END as ctr,
        c.word_count
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' f
    JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
      ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_avg_position > 0
    GROUP BY f.content_hash_id, c.word_count
""").df()

features_df["word_count"] = features_df["word_count"].fillna(features_df["word_count"].median())

X = features_df[["avg_position", "impressions_total", "ctr", "word_count"]]
X_scaled = StandardScaler().fit_transform(X)

for k in [3, 4, 5, 6]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    score = silhouette_score(X_scaled, km.labels_, sample_size=10000, random_state=42)
    print(f"k={k}: silhouette={score:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

k=3: silhouette=0.447
k=4: silhouette=0.475
k=5: silhouette=0.478
k=6: silhouette=0.482


In [19]:
final_km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_scaled)
features_df["cluster"] = final_km.labels_

cluster_profile = features_df.groupby("cluster")[["avg_position", "impressions_total", "ctr", "word_count"]].mean()
cluster_profile["n_pages"] = features_df["cluster"].value_counts().sort_index()
print(cluster_profile)

         avg_position  impressions_total       ctr   word_count  n_pages
cluster                                                                 
0           51.782805         536.034035  0.000936  2839.871232    29029
1           10.134531        1224.174893  0.003157  2715.318906   143105
2           11.573453       32219.692363  0.002875  3003.632565     2776
3            8.623061           2.276650  0.625453  1464.373096      394


### Cluster profiles (k=4)

- **Cluster 0:"Steady Middle"** (143,110 pages): page-1 adjacent (pos 10.1), moderate traffic (1,223 impressions), typical CTR (0.34%). The baseline "normal" page.
- **Cluster 1:"Suspicious Outliers"** (292 pages): near-perfect CTR (72.7%) but only 1-2 impressions almost certainly noise from tiny sample sizes, not genuine high performance. Flagged as a data-quality cluster, not an action archetype.
- **Cluster 2: "High-Traffic Workhorses"** (2,776 pages): massive impressions (32,220 avg), decent position (11.6), typical CTR. These are the site's real traffic drivers protect, don't disturb.
- **Cluster 3: "Buried and Struggling"** (29,126 pages): deep average position (51.7 — page 5+), lowest CTR (0.10%) of any real cluster. Likely genuine improvement candidates visible enough to matter, ranked too low to perform.

In [20]:
baseline_df = con.execute("""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) as avg_position,
        SUM(gsc_impressions) as impressions_total,
        CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks)*1.0/SUM(gsc_impressions) ELSE NULL END as ctr
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE gsc_avg_position > 0
    GROUP BY content_hash_id
""").df()

visible = (baseline_df["impressions_total"] >= 500).astype(int)
good_position = (baseline_df["avg_position"] <= 10).astype(int)
low_ctr = (baseline_df["ctr"].fillna(0) < 0.003).astype(int)
baseline_df["score"] = visible * good_position * low_ctr * baseline_df["impressions_total"]

print("Baseline regenerated:", len(baseline_df), "rows,", (baseline_df["score"]>0).sum(), "flagged")
flagged_ids = set(baseline_df[baseline_df["score"] > 0]["content_hash_id"])

features_df["flagged_by_baseline"] = features_df["content_hash_id"].isin(flagged_ids)

concentration = features_df.groupby("cluster")["flagged_by_baseline"].agg(["sum", "count", "mean"])
concentration.columns = ["n_flagged", "n_total", "pct_flagged"]
print(concentration)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline regenerated: 175304 rows, 23324 flagged
         n_flagged  n_total  pct_flagged
cluster                                 
0                0    29029     0.000000
1            22188   143105     0.155047
2             1136     2776     0.409222
3                0      394     0.000000


## Errors and interpretation

**Where the model is "wrong":** the worst fitting page (content_eadb33b5df496f4a, cluster 3) has 617,124 impressions more than double any other top-5 outlier and a distance-to-center roughly 3x higher than its neighbors. This page's traffic is so extreme that it doesn't fit cleanly into any cluster; K-Means places it in the "high-traffic" group by default, but it's really an outlier even within that group. The other worst-fits sit ambiguously between the "high-traffic" and "steady middle" archetypes solid but not extreme performers who could reasonably belong to either.

**What the clustering leans on:** CTR has by far the highest cluster-center spread (10.5), well above impressions (2.9), position (1.1), and word count (0.7). CTR is effectively the dominant axis separating these clusters.

**Sanity check:** this isn't suspiciously perfect or leakage-flavored  there's no future/label information involved  but it does mean the clustering is substantially "CTR driven," with position, traffic, and word count acting as secondary texture rather than equal contributors. This is worth stating honestly rather than implying all four features weighed in evenly.

In [21]:
import numpy as np

# Distance from each point to its own cluster center = how "wrong"/poor-fit that assignment is
distances = final_km.transform(X_scaled)
features_df["dist_to_own_center"] = distances[np.arange(len(features_df)), features_df["cluster"]]

# Show the 3 worst-fitting pages per cluster
worst_fits = features_df.sort_values("dist_to_own_center", ascending=False).head(5)
print(worst_fits[["content_hash_id", "cluster", "avg_position", "impressions_total", "ctr", "word_count", "dist_to_own_center"]])

# Which feature has the most spread between cluster centers = most influential in separating groups
centers_df = pd.DataFrame(final_km.cluster_centers_, columns=["avg_position", "impressions_total", "ctr", "word_count"])
print("\nCluster center spread per feature (higher = more influential in forming clusters):")
print(centers_df.std().sort_values(ascending=False))

                 content_hash_id  cluster  avg_position  impressions_total  \
54095   content_eadb33b5df496f4a        2      2.383011           617124.0   
40197   content_ec2e0346994fb5a5        2      2.854514           245276.0   
120154  content_e8a52cf3d5988c07        2     15.008339           244931.0   
39367   content_0e03de7680314cd5        2      2.675217           221310.0   
11888   content_44f34c0a90047651        2      7.346909           212404.0   

             ctr  word_count  dist_to_own_center  
54095   0.009185        2753          107.274950  
40197   0.006034        2581           39.077349  
120154  0.002731        3477           39.012135  
39367   0.003253        2784           34.680422  
11888   0.000113        3495           33.047571  

Cluster center spread per feature (higher = more influential in forming clusters):
ctr                  9.034263
impressions_total    2.904890
avg_position         1.138611
word_count           0.722338
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.